In [1]:
import pandas as pd
import requests

In [2]:
url1 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20150101:20150101'
api_sec = requests.get(url1)
datos_json1 = api_sec.json()
df = pd.DataFrame(datos_json1)
df.head()

,COD,Nombre,T3_Unidad,T3_Escala,Data
0,ADRH65783,"Acebeda, La sección 01001. Dato base. Renta ne...",Euros,,"[{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T..."
1,ADRH65782,"Acebeda, La sección 01001. Dato base. Renta ne...",Euros,,"[{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T..."
2,ADRH9802887,"Acebeda, La sección 01001. Dato base. Media de...",Euros,,"[{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T..."
3,ADRH9499057,"Acebeda, La sección 01001. Dato base. Mediana ...",Euros,,"[{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T..."
4,ADRH9499055,"Acebeda, La sección 01001. Dato base. Renta br...",Porcentaje,,"[{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T..."


In [3]:
df_expandido = df.explode("Data").copy()
df_expandido["Año"] = df_expandido["Data"].str["Anyo"].astype("Int64")
df_expandido["Valor"] = df_expandido["Data"].str["Valor"].astype(float)
df_expandido.head()

,COD,Nombre,T3_Unidad,T3_Escala,Data,Año,Valor
0,ADRH65783,"Acebeda, La sección 01001. Dato base. Renta ne...",Euros,,"{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T3...",2015,NaN
1,ADRH65782,"Acebeda, La sección 01001. Dato base. Renta ne...",Euros,,"{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T3...",2015,NaN
2,ADRH9802887,"Acebeda, La sección 01001. Dato base. Media de...",Euros,,"{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T3...",2015,NaN
3,ADRH9499057,"Acebeda, La sección 01001. Dato base. Mediana ...",Euros,,"{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T3...",2015,NaN
4,ADRH9499055,"Acebeda, La sección 01001. Dato base. Renta br...",Porcentaje,,"{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T3...",2015,NaN


In [4]:
filtro_madrid = df_expandido["Nombre"].str.contains("^Madrid", case=False, na=False)
filtro_T3unidad = df_expandido["T3_Unidad"].str.strip() == "Euros"
df_sec_filtrado = df_expandido[filtro_madrid & filtro_T3unidad].copy()

In [5]:
separacion_nombre = df_sec_filtrado["Nombre"].str.split(".", n=1, expand=True)
df_sec_filtrado["Seccion madrid"] = separacion_nombre[0].str.strip()
df_sec_filtrado["Dato base"] = separacion_nombre[1].str.strip()

In [6]:
indicadores_datobase = ["Dato base. Renta neta media por persona.","Dato base. Renta neta media por hogar."]
filtro_rentas_sec = df_sec_filtrado["Dato base"].isin(indicadores_datobase)
df_sec_filtrado = df_sec_filtrado[filtro_rentas_sec]
df_sec_filtrado['Seccion censal'] = df_sec_filtrado['Seccion madrid'].str[-3:]
df_sec_filtrado['Distrito madrid'] = df_sec_filtrado['Seccion madrid'].str[-5:-3].str.strip()
df_sec_filtrado.head()

,COD,Nombre,T3_Unidad,T3_Escala,Data,Año,Valor,Seccion madrid,Dato base,Seccion censal,Distrito madrid
6648,ADRH20975,Madrid sección 01001. Dato base. Renta neta me...,Euros,,"{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T3...",2015,22948.0,Madrid sección 01001,Dato base. Renta neta media por persona.,001,01
6649,ADRH20974,Madrid sección 01001. Dato base. Renta neta me...,Euros,,"{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T3...",2015,49363.0,Madrid sección 01001,Dato base. Renta neta media por hogar.,001,01
6654,ADRH20973,Madrid sección 01002. Dato base. Renta neta me...,Euros,,"{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T3...",2015,15045.0,Madrid sección 01002,Dato base. Renta neta media por persona.,002,01
6655,ADRH20972,Madrid sección 01002. Dato base. Renta neta me...,Euros,,"{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T3...",2015,29040.0,Madrid sección 01002,Dato base. Renta neta media por hogar.,002,01
6660,ADRH20971,Madrid sección 01003. Dato base. Renta neta me...,Euros,,"{'Fecha': '2015-01-01T00:00:00.000+01:00', 'T3...",2015,15177.0,Madrid sección 01003,Dato base. Renta neta media por persona.,003,01


In [7]:
columnas_drop1 = ["Data", "T3_Escala", "T3_Unidad", "Nombre", "Seccion madrid"]
df_madrid_seccion2015 = df_sec_filtrado.drop(columns=columnas_drop1).rename(columns={"Fecha": "Año"})
df_madrid_seccion2015.head()

,COD,Año,Valor,Dato base,Seccion censal,Distrito madrid
6648,ADRH20975,2015,22948.0,Dato base. Renta neta media por persona.,001,01
6649,ADRH20974,2015,49363.0,Dato base. Renta neta media por hogar.,001,01
6654,ADRH20973,2015,15045.0,Dato base. Renta neta media por persona.,002,01
6655,ADRH20972,2015,29040.0,Dato base. Renta neta media por hogar.,002,01
6660,ADRH20971,2015,15177.0,Dato base. Renta neta media por persona.,003,01


In [11]:
url2016 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20160101:20160101'
url2017 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20170101:20170101'
url2018 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20180101:20180101'
url2019 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20190101:20190101'
url2020 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20200101:20200101'
url2021 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20210101:20210101'
url2022 = 'https://servicios.ine.es/wstempus/js/es/DATOS_TABLA/31097?tip=A&tv=847:&date=20220101:20220101'

In [12]:
def procesar_datos_ine(url2016):
    api_sec = requests.get(url2016)
    datos_json = api_sec.json()
    df = pd.DataFrame(datos_json)

    df_expandido = df.explode("Data").copy()

    df_expandido["Año"] = df_expandido["Data"].str["Anyo"].astype("Int64")
    df_expandido["Valor"] = df_expandido["Data"].str["Valor"].astype(float)

    filtro_madrid = df_expandido["Nombre"].str.contains("^Madrid", case=False, na=False)
    filtro_T3unidad = df_expandido["T3_Unidad"].str.strip() == "Euros"
    df_sec_filtrado = df_expandido[filtro_madrid & filtro_T3unidad].copy()

    separacion_nombre = df_sec_filtrado["Nombre"].str.split(".", n=1, expand=True)
    df_sec_filtrado["Seccion madrid"] = separacion_nombre[0].str.strip()
    df_sec_filtrado["Dato base"] = separacion_nombre[1].str.strip()

    indicadores_datobase = ["Dato base. Renta neta media por persona.", "Dato base. Renta neta media por hogar.",]
    filtro_rentas_sec = df_sec_filtrado["Dato base"].isin(indicadores_datobase)
    df_sec_filtrado = df_sec_filtrado[filtro_rentas_sec]
    df_sec_filtrado["Seccion censal"] = df_sec_filtrado["Seccion madrid"].str[-3:]
    df_sec_filtrado["Distrito madrid"] = (df_sec_filtrado["Seccion madrid"].str[-5:-3].str.strip())

    columnas_drop1 = ["Data", "T3_Escala", "T3_Unidad", "Nombre", "Seccion madrid"]
    df_madrid_seccion = df_sec_filtrado.drop(columns=columnas_drop1).rename(columns={"Fecha": "Año"})

    return df_madrid_seccion

In [ ]:
df_madrid_seccion_2016 = procesar_datos_ine(url2016)
df_madrid_seccion_2017 = procesar_datos_ine(url2017)
df_madrid_seccion_2018 = procesar_datos_ine(url2018)
df_madrid_seccion_2019 = procesar_datos_ine(url2019)
df_madrid_seccion_2020 = procesar_datos_ine(url2020)
df_madrid_seccion_2021 = procesar_datos_ine(url2021)
df_madrid_seccion_2022 = procesar_datos_ine(url2022)

In [14]:
df_madrid_seccion_2016

,COD,Año,Valor,Dato base,Seccion censal,Distrito madrid
6648,ADRH20975,2016,22891.0,Dato base. Renta neta media por persona.,001,01
6649,ADRH20974,2016,48511.0,Dato base. Renta neta media por hogar.,001,01
6654,ADRH20973,2016,14608.0,Dato base. Renta neta media por persona.,002,01
6655,ADRH20972,2016,28573.0,Dato base. Renta neta media por hogar.,002,01
6660,ADRH20971,2016,16141.0,Dato base. Renta neta media por persona.,003,01
...,...,...,...,...,...,...
21589,ADRH40872,2016,55101.0,Dato base. Renta neta media por hogar.,032,21
21594,ADRH7226132,<NA>,NaN,Dato base. Renta neta media por persona.,033,21
21595,ADRH7226131,<NA>,NaN,Dato base. Renta neta media por hogar.,033,21
21600,ADRH10218871,<NA>,NaN,Dato base. Renta neta media por persona.,034,21


In [15]:
df_madrid_seccion_2017

,COD,Año,Valor,Dato base,Seccion censal,Distrito madrid
6648,ADRH20975,2017,23122.0,Dato base. Renta neta media por persona.,001,01
6649,ADRH20974,2017,48783.0,Dato base. Renta neta media por hogar.,001,01
6654,ADRH20973,2017,15211.0,Dato base. Renta neta media por persona.,002,01
6655,ADRH20972,2017,29753.0,Dato base. Renta neta media por hogar.,002,01
6660,ADRH20971,2017,17730.0,Dato base. Renta neta media por persona.,003,01
...,...,...,...,...,...,...
21589,ADRH40872,2017,57049.0,Dato base. Renta neta media por hogar.,032,21
21594,ADRH7226132,2017,15271.0,Dato base. Renta neta media por persona.,033,21
21595,ADRH7226131,2017,38786.0,Dato base. Renta neta media por hogar.,033,21
21600,ADRH10218871,<NA>,NaN,Dato base. Renta neta media por persona.,034,21


In [16]:
df_madrid_seccion_2018

,COD,Año,Valor,Dato base,Seccion censal,Distrito madrid
6648,ADRH20975,2018,27785.0,Dato base. Renta neta media por persona.,001,01
6649,ADRH20974,2018,58594.0,Dato base. Renta neta media por hogar.,001,01
6654,ADRH20973,2018,16464.0,Dato base. Renta neta media por persona.,002,01
6655,ADRH20972,2018,32928.0,Dato base. Renta neta media por hogar.,002,01
6660,ADRH20971,2018,17664.0,Dato base. Renta neta media por persona.,003,01
...,...,...,...,...,...,...
21589,ADRH40872,2018,60073.0,Dato base. Renta neta media por hogar.,032,21
21594,ADRH7226132,2018,15801.0,Dato base. Renta neta media por persona.,033,21
21595,ADRH7226131,2018,40105.0,Dato base. Renta neta media por hogar.,033,21
21600,ADRH10218871,<NA>,NaN,Dato base. Renta neta media por persona.,034,21


In [17]:
df_madrid_seccion_2019

,COD,Año,Valor,Dato base,Seccion censal,Distrito madrid
6648,ADRH20975,2019,31119.0,Dato base. Renta neta media por persona.,001,01
6649,ADRH20974,2019,66667.0,Dato base. Renta neta media por hogar.,001,01
6654,ADRH20973,2019,17113.0,Dato base. Renta neta media por persona.,002,01
6655,ADRH20972,2019,33286.0,Dato base. Renta neta media por hogar.,002,01
6660,ADRH20971,2019,19085.0,Dato base. Renta neta media por persona.,003,01
...,...,...,...,...,...,...
21589,ADRH40872,2019,64564.0,Dato base. Renta neta media por hogar.,032,21
21594,ADRH7226132,2019,16796.0,Dato base. Renta neta media por persona.,033,21
21595,ADRH7226131,2019,43504.0,Dato base. Renta neta media por hogar.,033,21
21600,ADRH10218871,<NA>,NaN,Dato base. Renta neta media por persona.,034,21


In [18]:
df_madrid_seccion_2020

,COD,Año,Valor,Dato base,Seccion censal,Distrito madrid
6648,ADRH20975,2020,25059.0,Dato base. Renta neta media por persona.,001,01
6649,ADRH20974,2020,52605.0,Dato base. Renta neta media por hogar.,001,01
6654,ADRH20973,2020,17175.0,Dato base. Renta neta media por persona.,002,01
6655,ADRH20972,2020,33052.0,Dato base. Renta neta media por hogar.,002,01
6660,ADRH20971,2020,18631.0,Dato base. Renta neta media por persona.,003,01
...,...,...,...,...,...,...
21589,ADRH40872,2020,61526.0,Dato base. Renta neta media por hogar.,032,21
21594,ADRH7226132,2020,17134.0,Dato base. Renta neta media por persona.,033,21
21595,ADRH7226131,2020,44444.0,Dato base. Renta neta media por hogar.,033,21
21600,ADRH10218871,<NA>,NaN,Dato base. Renta neta media por persona.,034,21


In [19]:
df_madrid_seccion_2021

,COD,Año,Valor,Dato base,Seccion censal,Distrito madrid
6648,ADRH20975,2021,24134.0,Dato base. Renta neta media por persona.,001,01
6649,ADRH20974,2021,51617.0,Dato base. Renta neta media por hogar.,001,01
6654,ADRH20973,2021,19101.0,Dato base. Renta neta media por persona.,002,01
6655,ADRH20972,2021,37684.0,Dato base. Renta neta media por hogar.,002,01
6660,ADRH20971,2021,18421.0,Dato base. Renta neta media por persona.,003,01
...,...,...,...,...,...,...
21589,ADRH40872,2021,63640.0,Dato base. Renta neta media por hogar.,032,21
21594,ADRH7226132,2021,12877.0,Dato base. Renta neta media por persona.,033,21
21595,ADRH7226131,2021,32180.0,Dato base. Renta neta media por hogar.,033,21
21600,ADRH10218871,<NA>,NaN,Dato base. Renta neta media por persona.,034,21


In [20]:
df_madrid_seccion_2022

,COD,Año,Valor,Dato base,Seccion censal,Distrito madrid
6648,ADRH20975,2022,26032.0,Dato base. Renta neta media por persona.,001,01
6649,ADRH20974,2022,55934.0,Dato base. Renta neta media por hogar.,001,01
6654,ADRH20973,2022,18878.0,Dato base. Renta neta media por persona.,002,01
6655,ADRH20972,2022,38385.0,Dato base. Renta neta media por hogar.,002,01
6660,ADRH20971,2022,19309.0,Dato base. Renta neta media por persona.,003,01
...,...,...,...,...,...,...
21589,ADRH40872,2022,69852.0,Dato base. Renta neta media por hogar.,032,21
21594,ADRH7226132,2022,14174.0,Dato base. Renta neta media por persona.,033,21
21595,ADRH7226131,2022,34040.0,Dato base. Renta neta media por hogar.,033,21
21600,ADRH10218871,2022,16796.0,Dato base. Renta neta media por persona.,034,21
